# BudgetRoute: free-Colab development pilot
## Goal
Test feasibility of pinned prompt-only routing, not a confirmatory quality/speed claim. Select a **free GPU** in Runtime → Change runtime type first. If no free GPU is available, stop. No Drive mount, API key, paid upgrade, public listener or tunnel is needed.
## Key assumptions
200 new MMLU development questions, frozen 120/40/40 roles; 40 diagnostic questions cannot certify a two-point margin. Models run greedy non-thinking FP16, 2,048 input / 64 output tokens, both resident. Save both the evidence bundle and this executed notebook before the runtime disappears.

## Setup — exact reviewed source
Paste the reviewed feature commit SHA. `primary` uses Qwen3.5; use the prespecified `fallback` only after retaining the primary failure report, in a separate workspace. Setup installs the exact source and Transformers 5.20.2; the full resolved installed-version inventory is frozen before measurement.

In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

SOURCE_SHA = input("Exact reviewed 40-character source SHA: ").strip()
assert re.fullmatch(r"[a-f0-9]{40}", SOURCE_SHA), "An explicit immutable reviewed SHA is required"
PROFILE = input("Profile: primary or fallback [primary]: ").strip() or "primary"
assert PROFILE in {"primary", "fallback"}
SOURCE = Path("/content/budgetroute-pilot-source")
if not SOURCE.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/MedvAx-AI/budgetroute-llm.git", str(SOURCE)],
        check=True,
    )
if subprocess.check_output(["git", "status", "--porcelain"], cwd=SOURCE, text=True).strip():
    raise RuntimeError("Source checkout is dirty; do not reset or overwrite it")
subprocess.run(["git", "fetch", "origin", SOURCE_SHA], cwd=SOURCE, check=True)
subprocess.run(["git", "switch", "--detach", SOURCE_SHA], cwd=SOURCE, check=True)
assert (
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SOURCE, text=True).strip()
    == SOURCE_SHA
)
os.chdir(SOURCE)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[all]", "transformers==5.20.2"],
    check=True,
)
os.environ["BUDGETROUTE_REVIEWED_SHA"] = SOURCE_SHA
WORKSPACE = Path("/content/budgetroute-pilot-" + PROFILE)
CONFIG = SOURCE / (
    "configs/benchmarks/colab-pilot.yaml"
    if PROFILE == "primary"
    else "configs/benchmarks/colab-pilot-fallback.yaml"
)

In [ ]:
from budgetroute.experiments.pilot import run_pilot_stage
from budgetroute.validation.colab import (
    run_component_checks,
    validate_colab_environment,
    validate_run_artifacts,
)

environment = validate_colab_environment()
print(
    {
        "source": SOURCE_SHA,
        "profile": PROFILE,
        "hardware": environment["hardware"],
        "role": "development pilot",
    }
)

## Data — freeze before fitting
Download only the pinned public dataset. Exclude the verified historical 500 questions and normalized duplicates. Persist exactly 200 questions and their fitting/selection/diagnostic roles. Remaining source questions are not evaluated.

In [ ]:
from budgetroute.experiments.pilot_data import prepare_pilot_data

splits = prepare_pilot_data(
    SOURCE / "configs/datasets/mmlu-pilot.yaml",
    SOURCE / "data/exclusions/mmlu-500-v1.json",
    WORKSPACE / "data",
)
print(
    {
        "fitting": len(splits.train_ids),
        "selection": len(splits.calibration_ids),
        "diagnostic": len(splits.test_ids),
        "data_sha256": splits.dataset_sha256,
    }
)

## Checks — actual remote components
This gate exercises real generators, confidence/batch order, MiniLM, FAISS HNSW, concurrency/batch grid, authenticated loopback API/feedback/metrics, memory and cleanup. A process exit code or a passed fake probe is not acceptance. If it fails, retain components.json; do not treat later stages as successful.

In [ ]:
component_report = run_component_checks(CONFIG, WORKSPACE)
print(
    {
        "components": json.loads(component_report.read_text())["status"],
        "report": str(component_report),
    }
)

## Experiment — baseline collection
Collect 400 paired outcomes with v2 cache checkpoints. Re-running resumes missing generations; cached quality evidence never substitutes for live timing.

In [ ]:
print(run_pilot_stage(CONFIG, WORKSPACE, stage="collect"))

### Fit and select
Fit numeric and semantic paired-quality gates on 120 fitting IDs, select empirical thresholds on 40 selection IDs. No certificate is requested. Infeasible gates always choose large. The confidence cascade remains secondary.

In [ ]:
print(run_pilot_stage(CONFIG, WORKSPACE, stage="fit"))

### Held-out pilot diagnostic and replay
Evaluate only the 40 diagnostic IDs. Matched-random assignments have exactly the semantic gate's cheap count. Report quality relative to always-large and absolute correctness separately.

In [ ]:
print(run_pilot_stage(CONFIG, WORKSPACE, stage="diagnostic"))

### Live timing
Three seeded policy-order blocks, 40 original questions per policy, one warmup per condition. Both generators stay resident in each block; no cache. Interrupted blocks restart completely. Repeats remain clustered by original question.

In [ ]:
print(run_pilot_stage(CONFIG, WORKSPACE, stage="timing"))

## Results — inspect, do not assume
The summary separates 200-question baselines, 40-question diagnostic quality and live timing. Its sample/runtime estimates are rough planning approximations, not a powered protocol or permission to start confirmation.

In [ ]:
summary_path = run_pilot_stage(CONFIG, WORKSPACE, stage="summarize")
summary = json.loads(summary_path.read_text())
print(
    json.dumps(
        {
            key: summary[key]
            for key in (
                "role",
                "baseline_questions",
                "diagnostic_questions",
                "paired",
                "live_paired",
                "rough_confirmation_planning",
                "recommendation",
                "limitations",
            )
        },
        indent=2,
    )
)

## Handoff — export before stopping
The curated bundle contains allowlisted evidence and hashes, not weights, caches, credentials or unrelated files. Download this executed notebook using File → Download → Download .ipynb as well. A prepared/unexecuted notebook is not remote evidence. Stop here; confirmation needs a separately approved frozen protocol.

In [ ]:
from google.colab import files

bundle = run_pilot_stage(CONFIG, WORKSPACE, stage="export")
print(validate_run_artifacts(WORKSPACE))
files.download(str(bundle))